In [0]:
# Configuration
from pyspark.sql import functions as F, Window
dbutils.widgets.combobox("catalog", "workspace", ["workspace", "dbr_dev"], "Unity Catalog")
dbutils.widgets.dropdown("run_checks", "true", ["true", "false"], "Verification / experimentation")
dbutils.widgets.combobox("silver_schema", "silver", ["silver", "gabrielajaniszews786_silver"], "Silver schema")
dbutils.widgets.combobox("gold_schema", "gold", ["gold", "gabrielajaniszews786_gold"], "Gold schema")
RUN_CHECKS = dbutils.widgets.get("run_checks") == "true"
CATALOG       = dbutils.widgets.get("catalog")
GOLD_SCHEMA = dbutils.widgets.get("gold_schema")
SILVER_SCHEMA = dbutils.widgets.get("silver_schema")

In [0]:
display(spark.sql(f"SELECT * FROM {CATALOG}.{SILVER_SCHEMA}.sensor_data LIMIT 40"))

In [0]:
display(spark.sql(f"SELECT * FROM {CATALOG}.{SILVER_SCHEMA}.sensor_data LIMIT 40"))

In [0]:
display(spark.sql(f"SELECT * FROM {CATALOG}.{SILVER_SCHEMA}.prices LIMIT 20"))

In [0]:
spark.sql(f"""
          CREATE TABLE IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}.dim_datacenter
          (
            dc_key BIGINT GENERATED ALWAYS AS IDENTITY,
            site_id STRING NOT NULL,
            site_name STRING,
            country STRING,
            bidding_zone STRING,
            valid_from TIMESTAMP NOT NULL,
            valid_to TIMESTAMP,
            is_current BOOLEAN NOT NULL,
            CONSTRAINT pk_dim_datacenter PRIMARY KEY (dc_key))
            """)

In [0]:
spark.sql(f"INSERT INTO {CATALOG}.{GOLD_SCHEMA}.dim_datacenter (site_id, site_name, country, bidding_zone, valid_from, valid_to, is_current) SELECT site_id, site_name, country, bidding_zone, valid_from, valid_to, is_current FROM {CATALOG}.{SILVER_SCHEMA}.dim_datacenter")
display(spark.sql(f"SELECT * FROM {CATALOG}.{GOLD_SCHEMA}.dim_datacenter"))

In [0]:
# Creating a fact table for events - hourly consumption
spark.sql(f"""
CREATE TABLE IF NOT EXISTS {CATALOG}.{GOLD_SCHEMA}.consumption_hourly (
  site_id               STRING     NOT NULL,
  bidding_zone          STRING     NOT NULL,
  date                  DATE       NOT NULL,
  hour                  INTEGER    NOT NULL,
  consumption_kwh       DECIMAL(10,4),
  avg_power_kw          DECIMAL(10,2),
  pue                   DECIMAL(4,3),
  cost_per_hour         DECIMAL(10,2)
)
USING DELTA
""")


In [0]:
spark.sql(f"""INSERT INTO {CATALOG}.{GOLD_SCHEMA}.consumption_hourly (site_id, date, hour, consumption_kwh, avg_power_kw, pue, cost_per_hour) 
          SELECT 
          s.site_id,
          date_trunc('hour', s.timestamp_utc) as date,
          hour(s.timestamp_utc) as hour,
          s.consumption_kwh,
          s.avg_power_kw,
          s.pue,
          s.consumption_kwh * p.price as cost_per_hour
          FROM {CATALOG}.{SILVER_SCHEMA}.sensor_data AS s
          LEFT JOIN {CATALOG}.{SILVER_SCHEMA}.prices AS p
          ON date_trunc('hour', s.timestamp_utc) = date_trunc('hour', p.timestamp_utc) AND s.bidding_zone = p.bidding_zone""")
display(spark.sql(f"SELECT * FROM {CATALOG}.{GOLD_SCHEMA}.consumption_hourly ORDER BY date, hour"))
